
## 1. Introducción

Esta libreta simula la **llegada incremental de interacciones, etiquetas y datos de uso** al sistema de producción, replicando el comportamiento de un entorno real donde los datos llegan de forma continua desde sistemas externos.

El mecanismo consiste en leer los archivos `data.json` almacenados en la carpeta **`source_buffer`** bajo el volumen **`landing_zone`**, que contienen los datos del futuro aún no procesados, y escribir sus registros en la estructura de directorios de **`events`**, respetando la **partición por año y mes**. Cada ejecución se escribe como un fichero independiente, de forma que el **`Auto Loader`** detecta cada fichero nuevo y lo ingiere de forma incremental sin necesidad de modificar ficheros existentes.

Las interacciones se inyectan en **orden cronológico estricto**. Para cada ventana temporal inyectada:
- Se copian las **etiquetas** cuyo `label_available_date` cae dentro de esa misma ventana, simulando el retraso real con el que los equipos de revisión confirman los casos.
- Se copian los registros de **usage** cuyo período mensual (`year_month`) ya ha concluido antes del final de la ventana inyectada, simulando la disponibilidad tardía de los datos de facturación.

La cantidad de datos inyectados en cada ejecución se controla mediante un único parámetro configurable:

* **`hours_to_inject`**: número de horas de datos del *buffer* que se inyectan en cada ejecución. Por ejemplo, `hours_to_inject = 3` inyecta todas las interacciones del *buffer* cuyo `timestamp` cae dentro de las 3 horas siguientes al último punto de continuación.

La simulación es **idempotente**: antes de comenzar, localiza automáticamente el **`timestamp` máximo** ya presente en `events/interactions` y omite todas las filas anteriores o iguales a ese valor, por lo que puede relanzarse sin duplicar datos en caso de fallo.


## 1. Importaciones y configuración

In [ ]:
exec(open("07_Utils.py").read(), globals())

In [ ]:
import json
import time
from calendar import monthrange
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
from pyspark.sql import functions as F

In [ ]:
# Base paths on the volume
landing_zone_path = Path("/") / "Volumes" / catalog / database / "landing_zone"

source_buffer_int_path   = landing_zone_path / "source_buffer" / "interactions"
source_buffer_lbl_path   = landing_zone_path / "source_buffer" / "labels"
source_buffer_usage_path = landing_zone_path / "source_buffer" / "usage"

events_int_path   = landing_zone_path / "events" / "interactions"
events_lbl_path   = landing_zone_path / "events" / "labels"
events_usage_path = landing_zone_path / "events" / "usage"

# Hours of interaction data to inject in this run
dbutils.widgets.text("hours_to_inject", "3")
hours_to_inject = int(dbutils.widgets.get("hours_to_inject"))

print(f"Source buffer (interactions) : {source_buffer_int_path}")
print(f"Source buffer (labels)       : {source_buffer_lbl_path}")
print(f"Source buffer (usage)        : {source_buffer_usage_path}")
print(f"Events (interactions)        : {events_int_path}")
print(f"Events (labels)              : {events_lbl_path}")
print(f"Events (usage)               : {events_usage_path}")
print(f"Hours to inject              : {hours_to_inject}")


## 2. Punto de continuación

Antes de comenzar la simulación se determina desde qué punto debe continuar, consultando el `timestamp` máximo ya presente en `events/interactions` mediante una lectura distribuida con **`Spark`**. Todas las interacciones del *buffer* con `timestamp` estrictamente posterior a ese valor serán las candidatas a copiar.

In [ ]:
def _find_latest_events_timestamp():
    """
    Query the maximum `timestamp` already present in `events/interactions`
    and return it as a `datetime` object.

    Returns `None` when no interactions have been copied yet.
    """
    try:
        max_ts = (
            spark.read
                 .json(str(events_int_path / "*" / "*" / "*.json"))
                 .agg(F.max("timestamp"))
                 .first()[0]
        )
        return datetime.fromisoformat(max_ts) if max_ts else None
    except Exception:
        return None


resume_from = _find_latest_events_timestamp()

if resume_from is None:
    print("No prior interactions found. Simulation will start from the beginning of the buffer.")
else:
    print(f"Resuming from timestamp: {resume_from.isoformat()}")
    print("Only interactions strictly after this timestamp will be copied.")


## 3. Carga y ordenación del *buffer* de interacciones

Se leen todos los archivos `data.json` de `source_buffer/interactions` mediante una lectura distribuida con **`Spark`**, se fusionan en una única lista ordenada cronológicamente por **`timestamp`** y se filtran las filas ya procesadas según el punto de continuación determinado en la sección anterior.

In [ ]:
def _load_buffer(base_path):
    """
    Extract `_year` and `_month` from the partition path of each record
    and return the full buffer as a flat list of dicts for in-memory processing.
    """
    return (
        spark.read
             .json(str(base_path / "*" / "*" / "data.json"))
             .withColumn("_year",  F.element_at(F.split(F.col("_metadata.file_path"), "/"), -3))
             .withColumn("_month", F.element_at(F.split(F.col("_metadata.file_path"), "/"), -2))
             .toPandas()
             .to_dict("records")
    )


all_int = _load_buffer(source_buffer_int_path)
for row in all_int:
    row["_ts"] = datetime.fromisoformat(str(row["timestamp"]))
all_int.sort(key=lambda row: row["_ts"])
print(f"Total rows in buffer: {len(all_int):,}")

if resume_from is not None:
    cutoff      = resume_from + timedelta(hours=hours_to_inject)
    pending_int = [row for row in all_int if resume_from < row["_ts"] <= cutoff]
else:
    cutoff      = all_int[0]["_ts"] + timedelta(hours=hours_to_inject)
    pending_int = [row for row in all_int if row["_ts"] <= cutoff]

print(f"Rows in buffer               : {len(all_int):,}")
print(f"Rows selected for injection  : {len(pending_int):,}")
print(f"Rows skipped (past or future): {len(all_int) - len(pending_int):,}")


## 4. Carga del *buffer* de etiquetas

Se leen todos los archivos `data.json` de `source_buffer/labels` con la misma estrategia que las interacciones. Solo se consideran las etiquetas cuyo **`label_available_date`** no es nulo, ya que las restantes corresponden a casos aún no resueltos por los equipos de revisión.

In [ ]:
all_lbl = _load_buffer(source_buffer_lbl_path)
print(f"Total rows in label buffer: {len(all_lbl):,}")

for row in all_lbl:
    lad = row.get("label_available_date")
    row["_lad"] = datetime.fromisoformat(lad) if lad else None

available_lbl = [row for row in all_lbl if row["_lad"] is not None]
available_lbl.sort(key=lambda row: row["_lad"])

# Track which labels have already been copied
copied_label_ids = set()

print(f"Labels with valid available date        : {len(available_lbl):,}")
print(f"Labels with null available date (skipped): {len(all_lbl) - len(available_lbl):,}")


## 5. Carga del *buffer* de usage

Se leen todos los archivos `data.json` de `source_buffer/usage`. A diferencia de las interacciones y etiquetas, el usage no tiene un `timestamp` puntual sino un período mensual (`year_month`). Un registro de usage se considera disponible cuando la simulación ha alcanzado o superado el **último día de ese mes**, replicando la disponibilidad tardía real de los datos de facturación.

La deduplicación entre sesiones se realiza consultando los pares `(customer_id, year_month)` ya presentes en `events/usage`.

In [ ]:
def _last_day_of_month(ym_str):
    """
    Return the last instant of the month as a datetime.
    '2025-01'  →  datetime(2025, 1, 31, 23, 59, 59)
    """
    year, month = int(ym_str[:4]), int(ym_str[5:7])
    last_day = monthrange(year, month)[1]
    return datetime(year, month, last_day, 23, 59, 59)


def _find_copied_usage_keys():
    """
    Load already-injected (customer_id, year_month) pairs from events/usage.
    Provides idempotency across session restarts.
    """
    try:
        df = spark.read.json(str(events_usage_path / "*" / "*" / "*.json"))
        return set(
            (row["customer_id"], row["year_month"])
            for row in df.select("customer_id", "year_month").collect()
        )
    except Exception:
        return set()


all_usage = _load_buffer(source_buffer_usage_path)
for row in all_usage:
    row["_period_end"] = _last_day_of_month(row["year_month"])
all_usage.sort(key=lambda row: row["_period_end"])

# Seed dedup set from what is already in events/usage
copied_usage_keys = _find_copied_usage_keys()

print(f"Total usage records in buffer          : {len(all_usage):,}")
print(f"Usage records already injected (dedup) : {len(copied_usage_keys):,}")


## 6. Inyección de datos

Cada ejecución inyecta:
1. Todas las **interacciones** cuyo `timestamp` cae dentro de la ventana `hours_to_inject` horas desde el punto de continuación.
2. Las **etiquetas** cuyo `label_available_date` cae dentro de esa misma ventana.
3. Los registros de **usage** cuyo período mensual ha concluido antes del último timestamp inyectado.

Todos los tipos se agrupan por partición `(year, month)` y se escriben como ficheros independientes para que el **`Auto Loader`** los detecte e ingiera de forma incremental.

In [ ]:
def _write_json(dest_path, records):
    """
    Write `records` to `dest_path` on the volume in newline-delimited
    JSON format (one record per line).
    """
    lines = "\n".join(json.dumps(record, default=str) for record in records)
    dbutils.fs.put(dest_path, lines, overwrite=True)


def _clean_row(row):
    """
    Remove internal metadata keys added during loading before writing to disk.
    """
    return {key: value for key, value in row.items() if not key.startswith("_")}


### 6.1. Copia de etiquetas por ventana temporal

Por cada lote de interacciones, se seleccionan las etiquetas cuyo **`label_available_date`** cae dentro de la ventana temporal del lote y que aún no han sido copiadas en iteraciones anteriores. Las etiquetas se agrupan por partición de destino `(year, month)` y se escriben en **`events/labels`** como un fichero independiente por lote.

In [ ]:
def _copy_labels(window_start, window_end, batch_timestamp):
    """
    Copy all labels whose `label_available_date` falls within
    [window_start, window_end] and have not yet been copied.

    Returns the number of labels written.
    """
    to_copy = [
        row for row in available_lbl
        if window_start <= row["_lad"] <= window_end
        and row["transaction_id"] not in copied_label_ids
    ]
    if not to_copy:
        return 0

    partitions = {}
    for row in to_copy:
        key = (row["_year"], row["_month"])
        partitions.setdefault(key, []).append(row)

    for (year, month), rows in partitions.items():
        dest_path = str(events_lbl_path / year / month / f"batch_{batch_timestamp}.json")
        _write_json(dest_path, [_clean_row(r) for r in rows])
        for row in rows:
            copied_label_ids.add(row["transaction_id"])

    return len(to_copy)


### 6.2. Copia de usage por período mensual

Se inyectan todos los registros de usage cuyo período mensual (`year_month`) ya ha concluido antes del último timestamp de la ventana inyectada. El criterio de deduplicación es el par `(customer_id, year_month)`: un cliente tiene un único registro de usage por mes, por lo que no pueden generarse duplicados aunque la celda se relance.

In [ ]:
def _copy_usage(window_end, batch_timestamp):
    """
    Inject all usage records whose billing period ended on or before
    `window_end` (the latest interaction timestamp in the current batch)
    and that have not yet been copied.

    Partitions by the year/month embedded in `year_month` (e.g. '2025-01'
    → year='2025', month='01') — not from the file path, since usage has
    no intra-month timestamp.

    Returns the number of usage records written.
    """
    to_copy = [
        row for row in all_usage
        if row["_period_end"] <= window_end
        and (row["customer_id"], row["year_month"]) not in copied_usage_keys
    ]
    if not to_copy:
        return 0

    partitions = {}
    for row in to_copy:
        year  = row["year_month"][:4]
        month = row["year_month"][5:7]
        partitions.setdefault((year, month), []).append(row)

    for (year, month), rows in partitions.items():
        dest_path = str(events_usage_path / year / month / f"batch_{batch_timestamp}.json")
        _write_json(dest_path, [_clean_row(r) for r in rows])
        for row in rows:
            copied_usage_keys.add((row["customer_id"], row["year_month"]))

    return len(to_copy)


### 6.3. Ejecución

Las interacciones de cada lote se agrupan por partición de destino `(year, month)` y se escriben en **`events/interactions`** como un fichero `.json` independiente. A continuación se copian las etiquetas y los registros de usage correspondientes a la misma ventana temporal.

In [ ]:
if not pending_int:
    print("No pending interactions for this time window.")
else:
    n_pending = len(pending_int)
    batch_ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
    header    = (
        f"Injecting {n_pending:,} interactions covering {hours_to_inject} hour(s) "
        f"from {pending_int[0]['_ts']} to {pending_int[-1]['_ts']}."
    )
    separator = "-" * len(header)

    print(header)
    print(separator)

    # 1. Copy interactions, grouped by (year, month) partition
    int_partitions = {}
    for row in pending_int:
        key = (row["_year"], row["_month"])
        int_partitions.setdefault(key, []).append(row)

    for (year, month), rows in int_partitions.items():
        dest_path = str(events_int_path / year / month / f"batch_{batch_ts}.json")
        _write_json(dest_path, [_clean_row(r) for r in rows])

    # 2. Copy labels whose available date falls within the injection window
    n_lbl   = _copy_labels(pending_int[0]["_ts"], pending_int[-1]["_ts"], batch_ts)

    # 3. Copy usage records whose monthly period ended before window_end
    n_usage = _copy_usage(pending_int[-1]["_ts"], batch_ts)

    print(separator)
    print("Injection complete.")
    print(f"  Interactions injected : {n_pending:,}")
    print(f"  Labels injected       : {n_lbl:,}")
    print(f"  Usage records injected: {n_usage:,}")


## 7. Conclusiones y siguientes pasos

### ¿Qué hace esta libreta?

1. **Continuación idempotente**: Antes de comenzar, consulta el `timestamp` máximo ya presente en `events/interactions` mediante una lectura distribuida con `Spark`, omitiendo todas las filas anteriores o iguales a ese valor para evitar duplicados en ejecuciones repetidas. Para `usage`, la deduplicación se basa en los pares `(customer_id, year_month)` ya presentes en `events/usage`.
2. **Inyección por ventana temporal**: Las interacciones se procesan en **orden cronológico estricto** y se escriben en `events` respetando la partición `year/month`, generando un fichero independiente por ejecución que el **`Auto Loader`** detecta e ingiere de forma incremental. El parámetro **`hours_to_inject`** controla cuántas horas de datos del buffer se inyectan en cada ejecución.
3. **Propagación simultánea de etiquetas**: Para cada ejecución, se copian las etiquetas cuyo **`label_available_date`** cae dentro de la ventana temporal inyectada, simulando el retraso real de confirmación.
4. **Inyección de usage por período mensual**: Se inyectan los registros de uso cuyo mes de facturación (`year_month`) ha concluido antes del último timestamp inyectado, simulando la disponibilidad tardía de los datos de consumo y facturación.

### ¿Cuándo ejecutar esta libreta?

Durante las primeras pruebas, esta libreta se ejecuta **manualmente** desde el entorno de desarrollo para validar el flujo completo de extremo a extremo. Una vez validado, se integra como tarea `Run_Simulation` en el trabajo **`Simulation Pipeline`**, que se ejecuta cada hora para inyectar automáticamente nuevas instancias en `events/` antes de que el pipeline `Medallion` las ingiera.

### ¿Qué sigue?

Tras cada ejecución, el pipeline `Medallion` ingiere los nuevos ficheros, publica las características actualizadas en el *online feature store* y genera predicciones sobre las interacciones recién llegadas.